# Assignment 2 

## Group name: ID2214 - 8

### Project members: 
Bernard Rumar, bernardr@kth.se \
Singvalayappa Velayutham, sinvel@kth.se \
Vivienne Curewitz, curewitz@kth.se

### Declaration:
By submitting this assignment, it is hereby declared that all group members listed above have contributed to the solution, either with code that appear in the final solution below, or with code that has been evaluated and compared to the final solution, but for some reason has been excluded. It is also declared that all project members fully understand all parts of the final solution and can explain it upon request.

It is furthermore declared that the code below is a contribution by the project members only, and specifically that no part of the solution has been copied from any other source (except for lecture slides at the course ID2214), no part of the solution has been provided by someone not listed as a project member above, and no part of the solution has been generated by a system such as a Generative AI tool, *other than where the assignment text explicitly allows such use*. 

It is furthermore declared that the submitted assignment will not be shared during the course, with any individual other than the group members listed above and teachers of the course ID2214. In particular, the assignment will not be uploaded to any public repository. The submitted assignment can be shared after the course only if written consent has been provided by the course responsible of ID2214.

It is furthermore declared that it has been understood that no other library/package than the Python 3 standard library, NumPy and pandas may be used in the solution for this assignment.

### Instructions
All parts of the assignment starting with number 1 below are mandatory. Sufficient (but not necessarily correct) solutions
on these parts will give 1 point (in total). The remaining parts (starting with number 2) are optional and will not give additional points. 

Note that you do not have to develop the code directly within the notebook
but may instead copy the comments and test cases to a more convenient development environment
and when everything works as expected, you may paste your functions into this
notebook, do a final testing (all cells should succeed), save the notebook including the output
and submit the whole notebook (a single file) in Canvas; do not forget to fill in your group number
and names above, thereby confirming that you compy with the above declaration. 

**Read and confirm the declaration before starting!** 

## Load NumPy, pandas and time

In [1]:
import numpy as np
import pandas as pd
import time

In [2]:
from platform import python_version

print(f"Python version: {python_version()}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

Python version: 3.11.7
NumPy version: 2.4.6
Pandas version: 3.1.0


## Reused functions from Assignment 1

In [3]:
# Copy and paste functions from Assignment 1 here that you need for this assignment
# Copy/paste is a terrible way to do code reuse, 
def create_column_filter(df: pd.DataFrame):
    drop = [col for col in df.keys() if df.nunique(dropna=True)[col] <= 1 and col != "ID" and col != "CLASS"]
    ret_df = df.drop(drop, axis=1)
    #print(drop)
    return ret_df, ret_df.keys()

def apply_column_filter(df: pd.DataFrame, column_filter):
    return df.select(column_filter)
df = pd.DataFrame({"CLASS":[1,0,1,0,1],"A":[1,2,np.nan,4,5],"B":[1,1,1,1,np.nan],"C":["h","h",np.nan,"i","h"],"D":[np.nan,np.nan,np.nan,np.nan,np.nan]})


def create_normalization(df: pd.DataFrame, normalizationtype="minmax"):
    triple_map = {}
    new_df = df.copy()
    if normalizationtype == "minmax":            
        for col in df.keys():
            if col == "CLASS" or col == "ID" or df[col].dtype != "float64" and df[col].dtype != "int64":
                continue
            min_value, max_value = df[col].min(), df[col].max()
            triple_map[col] = ("minmax", min_value, max_value)
            new_df[col] = (df[col] - min_value) / (max_value - min_value)
    elif normalizationtype == "zscore":
        for col in df.keys():
            if col == "CLASS" or col == "ID":
                continue
            mean, std = df[col].mean(), df[col].std()
            triple_map[col] = ("zscore", mean, std)
            new_df[col] = new_df[col].apply(lambda x: (x - mean)/std)
    else:
        raise Exception("normalization type not supported")
    return new_df, triple_map
        
def apply_normalization(df, norm):
    ret_df = df.copy()
    for col, (ntype, v1, v2) in norm.items():
        if ntype == "minmax":
            ret_df[col] = ret_df[col].apply(lambda x: (x - v1)/(v2 - v1))
        elif ntype == "zscore":
            ret_df[col] = ret_df[col].apply(lambda x: (x-v1)/v2)
    return ret_df

def create_imputation(df:pd.DataFrame):
    map_values = {}
    new_df = df.copy()
    for col in new_df.keys():
        if (col != "CLASS" and col != "ID") and (new_df[col].dtype == "float64" or new_df[col].dtype == "int64"):
            if (new_df[col].values.size == 0 or new_df[col].isna().all()):
                map_values[col] = 0
                new_df[col] = new_df[col].fillna(0)
            else:
                mean = new_df[col].mean()
                new_df[col] = new_df[col].fillna(mean if new_df[col].dtype == "float64" else int(mean))
                map_values[col] = mean if new_df[col].dtype == "float64" else int(mean)

        elif (col != "CLASS" and col != "ID") and (new_df[col].dtype == "str" or new_df[col].dtype == "object" or new_df[col].dtype == "category"):
            if (new_df[col].values.size == 0 or new_df[col].isna().all()):
                if new_df[col].dtype == "category":
                    map_values[col] = new_df[col].cat.categories[0]
                    new_df[col] = new_df[col].fillna(new_df[col].cat.categories[0])
                else:
                    map_values[col] = ""
                    new_df[col] = new_df[col].fillna("")
            else:
                mode = new_df[col].mode()[0]
                new_df[col] = new_df[col].fillna(mode)
                map_values[col] = mode 
        print(f" the dtype of this column {col} is {new_df[col].dtype}")
    return new_df, map_values 

def apply_imputation(df: pd.DataFrame, imputation):
    new_df = df.copy()
    for col in new_df.columns:
        if col in imputation:
            new_df[col] = new_df[col].fillna(imputation[col])
    return new_df

# updated so that these do not encode numerical values
def create_one_hot(df):
    cat_cols = [c for c in list(set(df.columns) - {'ID', 'CLASS'}) if  not pd.api.types.is_numeric_dtype(df[c])]
    one_hot = {k: df[k].unique() for k in cat_cols}

    return apply_one_hot(df, one_hot), one_hot

def apply_one_hot(df, one_hot):
    nd = df.copy()
    print(one_hot.keys())
    for col, cats in one_hot.items():
        for cat in cats:
            nd[f"{col}_{cat}"] = [float(v == cat) for v in df[col]]

    return nd.drop(columns=one_hot.keys())


def accuracy(df, correctlabels):
    return sum(df.idxmax(axis='columns') == correctlabels)/len(df)

## 1. Define the class kNN

### Required: Original solution, must not have been generated with support of Generative AI

In [25]:
# Define the class kNN with three functions __init__, fit and predict (after the comments):
#
# Input to __init__: 
# self - the object itself
#
# Output from __init__:
# <nothing>
# 
# This function does not return anything but just initializes the following attributes of the object (self) to None:
# column_filter, imputation, normalization, one_hot, labels, training_labels, training_data, training_time
class kNN:
    def __init__(self):
        self.column_filter = None
        self.imputation = None
        self.normalization = None
        self.one_hot = None
        self.labels = None
        self.training_labels = None
        self.training_data = None
        self.training_time = None

# Input to fit:
# self              - the object itself
# df                - a dataframe (where the column names "CLASS" and "ID" have special meaning)
# normalizationtype - "minmax" (default) or "zscore"
#
# Output from fit:
# <nothing>
#
# The result of applying this function should be:
#
# self.column_filter   - a column filter (see Assignment 1) from df
# self.imputation      - an imputation mapping (see Assignment 1) from df
# self.normalization   - a normalization mapping (see Assignment 1), using normalizationtype from the imputed df
# self.one_hot         - a one-hot mapping (see Assignment 1)
# self.training_labels - a pandas series corresponding to the "CLASS" column, set to be of type "category" 
# self.labels          - a list of the categories (class labels) of the previous series
# self.training_data   - the values (an ndarray) of the transformed dataframe, i.e., after employing imputation, 
#                        normalization, and possibly one-hot encoding, and also after removing the "CLASS" and "ID" columns
#
# Note that the function does not return anything but just assigns values to the attributes of the object.
#
    def fit(self, df: pd.DataFrame, noramlizationtype="minmax"):
        # preprocessing
        df, self.column_filter = create_column_filter(df)
        df, self.imputation    = create_imputation(df)
        df, self.normalization = create_normalization(df, normalizationtype=noramlizationtype)
        df, self.one_hot       = create_one_hot(df)

        # labels
        self.training_labels = df['CLASS'].astype('category')
        self.labels = list(self.training_labels.cat.categories)
        #print(f"count class:\n{self.training_labels.value_counts()}")
        # data
        self.training_data = df.drop(columns=['CLASS', 'ID'], errors='ignore').to_numpy() # ignore errors when column doesn't exist

        print(f'training labels:\t{self.training_labels}\nlabels:\t{self.labels}')
        #print(f'data:\t{self.training_data}')

# Input to predict:
# self - the object itself
# df   - a dataframe
# k    - an integer >= 1 (default = 5)
# 
# Output from predict:
# predictions - a dataframe with class labels as column names and the rows corresponding to
#               predictions with estimated class probabilities for each row in df, where the class probabilities
#               are estimated by the relative class frequencies in the set of class labels from the k nearest 
#               (with respect to Euclidean distance) neighbors in training_data
#
# Hint 1: Drop any "CLASS" and "ID" columns first and then apply column filtering, imputation, normalization and one-hot
#
# Hint 2: Get the numerical values (as an ndarray) from the resulting dataframe and iterate over the rows 
#         calling some sub-function, e.g., get_nearest_neighbor_predictions(x_test,k), which for a test row
#         (numerical input feature values) finds the k nearest neighbors and calculate the class probabilities.
#
# Hint 3: This sub-function may first find the distances to all training instances, e.g., pairs consisting of
#         training instance index and distance, and then sort them according to distance, and then (using the indexes
#         of the k closest instances) find the corresponding labels and calculate the relative class frequencies
    def predict(self, df: pd.DataFrame, k: int=5):

        
        # print(self.column_filter)
        df = apply_column_filter(df,self.column_filter)
        df = df.drop(columns=['CLASS', 'ID'], errors='ignore')
        df = apply_imputation(df,self.imputation)
        df = apply_normalization(df,self.normalization)
        df = apply_one_hot(df,self.one_hot)
        display(df)
        
        def get_nearest_neighbors(x_test, k):
            distances = []
            for i, x_train in enumerate(self.training_data):
                dist = np.sqrt(np.sum([(a-b)*(a-b) for a, b in zip(x_test, x_train)]))
                distances.append((dist,i))
            distances = sorted(distances)
            distanceLabels = [self.training_labels[index] for (_,index) in distances[0:k]]
            #print(distances)
            #print(distanceLabels)
            classFreq = {label:distanceLabels.count(label)/k for label in self.labels}
            #print(classFreq)
            return classFreq

       
        new_df = pd.DataFrame(columns=self.labels)
        rows = df.to_numpy()
        #print(f"rows are",rows)
        for i,row in enumerate(rows):
            r = get_nearest_neighbors(row, k)
            new_df.loc[i] = r
        return new_df


#glass_train_df = pd.read_csv("glass_train.csv")
#glass_test_df = pd.read_csv("glass_test.csv")
#KNN = kNN()
#KNN.fit(glass_train_df)
#KNN.predict(glass_test_df, k=5)

In [26]:
# Test your code (leave this part unchanged, except for if auc is undefined)

glass_train_df = pd.read_csv("anneal_train.csv")

glass_test_df = pd.read_csv("anneal_test.csv")

knn_model = kNN()

t0 = time.perf_counter()
knn_model.fit(glass_train_df)
print("Training time: {0:.2f} s.".format(time.perf_counter()-t0))

test_labels = glass_test_df["CLASS"]

k_values = [1,3,5,7,9]
results = np.empty((len(k_values),3))

for i in range(len(k_values)):
    t0 = time.perf_counter()
    predictions = knn_model.predict(glass_test_df,k=k_values[i])
    print("Testing time (k={0}): {1:.2f} s.".format(k_values[i],time.perf_counter()-t0))
    results[i] = [accuracy(predictions,test_labels)]#,brier_score(predictions,test_labels),
                  #auc(predictions,test_labels)] # Assuming that you have defined auc - remove otherwise

results = pd.DataFrame(results,index=k_values,columns=["Accuracy","Brier score","AUC"])

print()
display("results",results)

 the dtype of this column family is str
 the dtype of this column steel is str
 the dtype of this column carbon is int64
 the dtype of this column hardness is int64
 the dtype of this column condition is str
 the dtype of this column formability is float64
 the dtype of this column strength is int64
 the dtype of this column surface-quality is str
 the dtype of this column enamelability is float64
 the dtype of this column bw/me is str
 the dtype of this column blue-bright-varn-clean is str
 the dtype of this column shape is str
 the dtype of this column thick is float64
 the dtype of this column width is float64
 the dtype of this column len is int64
 the dtype of this column oil is str
 the dtype of this column bore is int64
 the dtype of this column CLASS is str
dict_keys(['blue-bright-varn-clean', 'oil', 'family', 'steel', 'condition', 'shape', 'bw/me', 'surface-quality'])
training labels:	0      3
1      3
2      5
3      3
4      U
      ..
444    3
445    3
446    3
447    3
448

,carbon,hardness,formability,strength,enamelability,thick,width,len,bore,blue-bright-varn-clean_B,...,condition_S,condition_A,shape_SHEET,shape_COIL,bw/me_B,bw/me_M,surface-quality_F,surface-quality_G,surface-quality_E,surface-quality_D
0,0.000000,0.0,0.250000,0.000000,0.714286,0.122074,0.865574,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.000000,0.0,0.250000,0.000000,0.714286,0.122340,0.865574,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,0.000000,0.0,0.312937,0.516667,0.714286,0.361702,0.400000,0.000000,0.833333,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,0.000000,0.0,0.250000,0.000000,0.714286,0.787234,0.098361,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.057143,0.0,0.312937,0.000000,0.714286,0.069149,0.016393,0.000000,0.000000,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444,0.000000,0.0,0.250000,0.000000,0.714286,0.015957,0.865574,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
445,0.785714,0.0,0.312937,0.000000,0.714286,0.335106,0.590164,0.000000,0.000000,1.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
446,0.000000,0.0,0.250000,0.000000,0.714286,0.148936,0.400000,1.000000,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
447,0.085714,0.0,0.312937,0.000000,0.714286,0.202128,0.400000,0.000000,0.000000,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0


Testing time (k=1): 4.70 s.
dict_keys(['blue-bright-varn-clean', 'oil', 'family', 'steel', 'condition', 'shape', 'bw/me', 'surface-quality'])


,carbon,hardness,formability,strength,enamelability,thick,width,len,bore,blue-bright-varn-clean_B,...,condition_S,condition_A,shape_SHEET,shape_COIL,bw/me_B,bw/me_M,surface-quality_F,surface-quality_G,surface-quality_E,surface-quality_D
0,0.000000,0.0,0.250000,0.000000,0.714286,0.122074,0.865574,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.000000,0.0,0.250000,0.000000,0.714286,0.122340,0.865574,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,0.000000,0.0,0.312937,0.516667,0.714286,0.361702,0.400000,0.000000,0.833333,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,0.000000,0.0,0.250000,0.000000,0.714286,0.787234,0.098361,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.057143,0.0,0.312937,0.000000,0.714286,0.069149,0.016393,0.000000,0.000000,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444,0.000000,0.0,0.250000,0.000000,0.714286,0.015957,0.865574,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
445,0.785714,0.0,0.312937,0.000000,0.714286,0.335106,0.590164,0.000000,0.000000,1.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
446,0.000000,0.0,0.250000,0.000000,0.714286,0.148936,0.400000,1.000000,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
447,0.085714,0.0,0.312937,0.000000,0.714286,0.202128,0.400000,0.000000,0.000000,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0


Testing time (k=3): 3.98 s.
dict_keys(['blue-bright-varn-clean', 'oil', 'family', 'steel', 'condition', 'shape', 'bw/me', 'surface-quality'])


,carbon,hardness,formability,strength,enamelability,thick,width,len,bore,blue-bright-varn-clean_B,...,condition_S,condition_A,shape_SHEET,shape_COIL,bw/me_B,bw/me_M,surface-quality_F,surface-quality_G,surface-quality_E,surface-quality_D
0,0.000000,0.0,0.250000,0.000000,0.714286,0.122074,0.865574,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.000000,0.0,0.250000,0.000000,0.714286,0.122340,0.865574,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,0.000000,0.0,0.312937,0.516667,0.714286,0.361702,0.400000,0.000000,0.833333,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,0.000000,0.0,0.250000,0.000000,0.714286,0.787234,0.098361,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.057143,0.0,0.312937,0.000000,0.714286,0.069149,0.016393,0.000000,0.000000,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444,0.000000,0.0,0.250000,0.000000,0.714286,0.015957,0.865574,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
445,0.785714,0.0,0.312937,0.000000,0.714286,0.335106,0.590164,0.000000,0.000000,1.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
446,0.000000,0.0,0.250000,0.000000,0.714286,0.148936,0.400000,1.000000,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
447,0.085714,0.0,0.312937,0.000000,0.714286,0.202128,0.400000,0.000000,0.000000,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0


... (3 outputs left out)

,carbon,hardness,formability,strength,enamelability,thick,width,len,bore,blue-bright-varn-clean_B,...,condition_S,condition_A,shape_SHEET,shape_COIL,bw/me_B,bw/me_M,surface-quality_F,surface-quality_G,surface-quality_E,surface-quality_D
0,0.000000,0.0,0.250000,0.000000,0.714286,0.122074,0.865574,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.000000,0.0,0.250000,0.000000,0.714286,0.122340,0.865574,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,0.000000,0.0,0.312937,0.516667,0.714286,0.361702,0.400000,0.000000,0.833333,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,0.000000,0.0,0.250000,0.000000,0.714286,0.787234,0.098361,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.057143,0.0,0.312937,0.000000,0.714286,0.069149,0.016393,0.000000,0.000000,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444,0.000000,0.0,0.250000,0.000000,0.714286,0.015957,0.865574,0.156148,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
445,0.785714,0.0,0.312937,0.000000,0.714286,0.335106,0.590164,0.000000,0.000000,1.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
446,0.000000,0.0,0.250000,0.000000,0.714286,0.148936,0.400000,1.000000,0.000000,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
447,0.085714,0.0,0.312937,0.000000,0.714286,0.202128,0.400000,0.000000,0.000000,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0


Testing time (k=9): 3.50 s.



'results'

,Accuracy,Brier score,AUC
1,0.926503,0.926503,0.926503
3,0.839644,0.839644,0.839644
5,0.824053,0.824053,0.824053
7,0.804009,0.804009,0.804009
9,0.792873,0.792873,0.792873


In [17]:
train_labels = glass_train_df["CLASS"]
predictions = knn_model.predict(glass_train_df,k=1)
print("Accuracy on training set (k=1): {0:.4f}".format(accuracy(predictions,train_labels)))
#print("AUC on training set (k=1): {0:.4f}".format(auc(predictions,train_labels)))
#print("Brier score on training set (k=1): {0:.4f}".format(brier_score(predictions,train_labels)))

dict_keys([])


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe
0,0.261785,0.241107,0.000000,0.327273,1.000000,0.434783,0.378378,0.000000,0.000000
1,0.288867,0.543478,0.775056,0.516364,0.504655,0.099839,0.276757,0.000000,0.000000
2,0.499498,0.709486,0.868597,0.196364,0.314711,0.000000,0.438919,0.000000,0.000000
3,0.213139,0.620553,0.000000,0.818182,0.636872,0.022544,0.430270,0.257143,0.027027
4,0.289870,0.448617,0.772829,0.545455,0.541899,0.106280,0.297297,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
102,0.404213,0.604743,0.536748,0.327273,0.549348,0.000000,0.469189,0.000000,0.000000
103,0.333501,0.547431,0.748330,0.487273,0.396648,0.091787,0.365405,0.000000,0.243243
104,0.257773,0.527668,0.777283,0.447273,0.528864,0.107890,0.286486,0.000000,0.270270
105,0.306921,0.290514,0.723831,0.316364,0.696462,0.099839,0.375135,0.000000,0.648649


Accuracy on training set (k=1): 1.0000


### Optional: Comment on assumptions, things that do not work properly, etc.


### Optional: Solution obtained using Generative AI

In [ ]:
# Insert code obtained using Generative AI here
# Note that this part is not mandatory

### Required (if solution by Generative AI has been provided above): Discussion

Answer the following questions below:

- What tool/model did you use?
- How did you instruct the model to generate a solution?
- What are the main differences between your solution and the one obtained using Generative AI?
- Which solution do you think would be the most easy to understand for someone outside your group? 

### Required (if solution by Generative AI has been provided above): Combined solution

In [ ]:
# Insert the solution that you would like to use in future projects; 
# it could either be one of the two solutions above or a combination of these
# Explain in a comment why you have chosen this combination

## 2. Define the class NaiveBayes

### Optional: Original solution, must not have been generated with support of Generative AI

In [ ]:
# Define the class NaiveBayes with three functions __init__, fit and predict (after the comments):
#
# Input to __init__: 
# self - the object itself
#
# Output from __init__:
# <nothing>
# 
# This function does not return anything but just initializes the following attributes of the object (self) to None:
# column_filter, binning, labels, class_priors, feature_class_value_counts, feature_class_counts
#
# Input to fit:
# self    - the object itself
# df      - a dataframe (where the column names "CLASS" and "ID" have special meaning)
# nobins  - no. of bins (default = 10)
# bintype - either "equal-width" (default) or "equal-size" 
#
# Output from fit:
# <nothing>
#
# The result of applying this function should be:
#
# self.column_filter              - a column filter (see Assignment 1) from df
# self.binning                    - a discretization mapping (see Assignment 1) from df
# self.class_priors               - a mapping (dictionary) from the labels (categories) of the "CLASS" column of df,
#                                   to the relative frequencies of the labels
# self.labels                     - a list of the categories (class labels) of the "CLASS" column of df
# self.feature_class_value_counts - a mapping from the feature (column name) to the number of
#                                   training instances with a specific combination of (non-missing, categorical) 
#                                   value for the feature and class label
# self.feature_class_counts       - a mapping from the feature (column name) to the number of
#                                   training instances with a specific class label and some (non-missing, categorical) 
#                                   value for the feature
#
# Note that the function does not return anything but just assigns values to the attributes of the object.
#
# Input to predict:
# self - the object itself
# df   - a dataframe
# 
# Output from predict:
# predictions - a dataframe with class labels as column names and the rows corresponding to
#               predictions with estimated class probabilities for each row in df, where the class probabilities
#               are estimated by the naive approximation of Bayes rule (see lecture slides)
#
# Hint 1: First apply the column filter and discretization
#
# Hint 2: Iterating over either columns or rows, and for each possible class label, calculate the relative
#         frequency of the observed feature value given the class (using feature_class_value_counts and 
#         feature_class_counts) 
#
# Hint 3: Calculate the non-normalized estimated class probabilities by multiplying the class priors to the
#         product of the relative frequencies
#
# Hint 4: Normalize the probabilities by dividing by the sum of the non-normalized probabilities; in case
#         this sum is zero, then set the probabilities to the class priors
#
# Hint 5: To clarify the assignment text a little: self.feature_class_value_counts should be a mapping from 
#         a column name (a specific feature) to another mapping, which given a class label and a value for 
#         the feature, returns the number of training instances which have included this combination, 
#         i.e., the number of training instances with both the specific class label and this value on the feature.
#
# Hint 6: As an additional hint, you may take a look at the slides from the NumPy and pandas lecture, to see how you 
#         may use "groupby" in combination with "size" to get the counts for combinations of values from two columns.




In [ ]:
# Test your code (leave this part unchanged, except for if auc is undefined)

glass_train_df = pd.read_csv("glass_train.csv")

glass_test_df = pd.read_csv("glass_test.csv")

nb_model = NaiveBayes()

test_labels = glass_test_df["CLASS"]

nobins_values = [3,5,10]
bintype_values = ["equal-width","equal-size"]
parameters = [(nobins,bintype) for nobins in nobins_values for bintype in bintype_values]

results = np.empty((len(parameters),3))

for i in range(len(parameters)):
    t0 = time.perf_counter()
    nb_model.fit(glass_train_df,nobins=parameters[i][0],bintype=parameters[i][1])
    print("Training time {0}: {1:.2f} s.".format(parameters[i],time.perf_counter()-t0))
    t0 = time.perf_counter()
    predictions = nb_model.predict(glass_test_df)
    print("Testing time {0}: {1:.2f} s.".format(parameters[i],time.perf_counter()-t0))
    results[i] = [accuracy(predictions,test_labels),brier_score(predictions,test_labels),
                  auc(predictions,test_labels)] # Assuming that you have defined auc - remove otherwise

results = pd.DataFrame(results,index=pd.MultiIndex.from_product([nobins_values,bintype_values]),
                       columns=["Accuracy","Brier score","AUC"])

print()
display("results",results)

In [ ]:
train_labels = glass_train_df["CLASS"]
nb_model.fit(glass_train_df)
predictions = nb_model.predict(glass_train_df)
print("Accuracy on training set: {0:.4f}".format(accuracy(predictions,train_labels)))
print("AUC on training set: {0:.4f}".format(auc(predictions,train_labels)))
print("Brier score on training set: {0:.4f}".format(brier_score(predictions,train_labels)))

### Optional: Comment on assumptions, things that do not work properly, etc.

### Optional: Solution obtained using Generative AI

In [ ]:
# Insert code obtained using Generative AI here
# Note that this part is not mandatory

### Required (if solution by Generative AI has been provided above): Discussion

Answer the following questions below:

- What tool/model did you use?
- How did you instruct the model to generate a solution?
- What are the main differences between your solution and the one obtained using Generative AI?
- Which solution do you think would be the most easy to understand for someone outside your group? 

### Required (if solution by Generative AI has been provided above): Combined solution

In [ ]:
# Insert the solution that you would like to use in future projects; 
# it could either be one of the two solutions above or a combination of these
# Explain in a comment why you have chosen this combination